# Intestine: full experiments with the selected settings

Both collagen-IV images, intensity baselines, joint-selected graph controls and
independently selected spatial comparisons. No configuration search is rerun.

**Planning runtime:** 40–90 minutes on an RTX 4070 SUPER; 1–3 hours on a standard
Colab T4, excluding setup/downloads. The notebook records actual stage times.
In Colab select **Runtime → Change runtime type → T4 GPU**.

Full execution trains 16 CNNs, 36 GraphMAEs and 24 native spatial comparisons.
The supplementary GIN comparison adds 10 GraphMAEs using the same CNN features.
Completed stages resume from `results/intestine`; keep this folder between sessions.

In [ ]:
from pathlib import Path
from urllib.request import urlopen

setup_file = Path("experiment_setup.py")
if not setup_file.exists():
    source = (
        "https://raw.githubusercontent.com/moeghaf/mantpy_reproducibility/"
        "main/notebooks/experiments/experiment_setup.py"
    )
    with urlopen(source, timeout=90) as response:
        setup_file.write_bytes(response.read())
from experiment_setup import prepare_experiment
prepare_experiment("intestine")

## Settings and output folder

The joint arm uses the selected CNN and edge-aware GINE configuration with
seeds 0–4. The matched arm uses an independently selected CNN with seeds 0–2;
all four spatial methods receive exactly the same features for each seed.
Both arms use 200 CNN epochs and 200 GraphMAE updates. Cluster numbers are arbitrary.

In [ ]:
import json
import time
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import anndata as ad
from scipy import sparse
from scipy.spatial import cKDTree
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from skimage.filters import threshold_multiotsu
import torch
import mantpy as mt
from intestine_helpers import (
    StageStore, PixelScorer, load_images, pixel_annotation, fit_selected_cnn,
    build_selected_graph, calculation_threads, seed_everything, array_sha256,
    plot_patch_labels, score_summary, write_json, environment, release_cuda,
    observe_graphmae_training,
)

SETTINGS = json.loads(Path("intestine_selected.json").read_text())
RESULTS = Path(os.environ.get("MANTPY_RESULTS", "results")) / "intestine"
RESULTS.mkdir(parents=True, exist_ok=True)
CACHE = os.environ.get("MANTPY_CACHE")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE != "cuda":
    raise RuntimeError("Select a T4 GPU runtime before running the full intestine fits.")
NOTEBOOK_STARTED = time.perf_counter()
print(environment())
pd.DataFrame([
    {"image": image, "K": config["k"], "patches": config["expected_nodes"],
     "joint CNN": f'{config["joint"]["cnn_width"]}-wide / {config["joint"]["cnn_readout"]}',
     "matched CNN": config["matched_cnn"]["architecture"]}
    for image, config in SETTINGS["images"].items()
])

## Load both images and their original annotations

Image 1 uses its five-layer 32-pixel annotation. Image 2 uses its native
four-class pixel annotation. The loaders verify the public data bundles.

In [ ]:
loaded = load_images(cache_dir=CACHE)
for image, item in loaded.items():
    annotation = pixel_annotation(image, item, item["image"].shape)
    figure, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(item["image"], cmap="gray",
                   vmax=SETTINGS["images"][image]["clip"])
    axes[0].set_title(f'{image}: collagen IV')
    axes[1].imshow(np.ma.masked_equal(annotation, 0), cmap="tab10",
                   vmin=1, vmax=SETTINGS["images"][image]["k"])
    axes[1].set_title("Original annotation")
    for axis in axes:
        axis.set_axis_off()
    figure.tight_layout()
    plt.show()
    item["annotation_for_scoring"] = annotation

## Apply the fixed foreground recipe

Clip each image at its positive-pixel 99th percentile and retain 32 × 32
patches above the original patch-mean threshold. The stored pixel threshold
belongs to the original preparation; evaluation uses annotated pixels within
these same foreground patches for every method.

In [ ]:
patches, scorers, stores = {}, {}, {}
coverage_rows = []
for image, item in loaded.items():
    config = SETTINGS["images"][image]
    raw = item["image"]
    clip = float(np.percentile(raw[raw > 0].astype(np.float32), 99))
    assert np.isclose(clip, config["clip"]), (image, clip)
    patches[image] = mt.pp.image_ecm_patches(
        raw, channel_names=["ColIV"], patch_size=32, clip_percentile=99,
        pixel_thresholds=[config["pixel_threshold"]],
        patch_thresholds=[config["patch_threshold"]],
        feature_families=("summary",), sample_id=image, random_state=0,
    )
    assert patches[image].n_obs == config["expected_nodes"]
    scorers[image] = PixelScorer(
        patches[image], item["annotation_for_scoring"], config["k"],
    )
    stores[image] = StageStore(
        RESULTS, image, array_sha256(raw) + ":" + array_sha256(patches[image].obs_names.to_numpy(str)),
    )
    pd.DataFrame({"patch_id": patches[image].obs_names,
                  "grid_y": patches[image].obs["grid_y"].to_numpy(),
                  "grid_x": patches[image].obs["grid_x"].to_numpy()}).to_csv(
        RESULTS / image / "patch_ids.csv", index=False,
    )
    coverage_rows.append({
        "image": image, "nodes": patches[image].n_obs,
        "coverage": scorers[image].coverage,
        "annotated_pixels": int(scorers[image].total.sum()),
        "covered_annotated_pixels": int(scorers[image].covered.sum()),
    })
coverage = pd.DataFrame(coverage_rows)
coverage.to_csv(RESULTS / "coverage.csv", index=False)
display(coverage)

## Pixel and patch-intensity baselines

Pixel K-means fits a fixed sample of two million native pixels, including
background, then labels the complete image. Patch-mean K-means and Multi-Otsu
use the retained patches. All three are scored on the same covered annotation.

In [ ]:
score_rows = []
def record_score(image, arm, method, seed, labels, pixel=False):
    score_rows.append({
        "image": image, "arm": arm, "method": method, "seed": seed,
        **scorers[image].score(labels, pixel=pixel),
    })

for image, item in loaded.items():
    config, store = SETTINGS["images"][image], stores[image]
    request = {"k": config["k"], "seed": 0, "n_init": "auto",
               "clip": config["clip"], "sample_pixels": 2_000_000}
    saved = store.load("intensity", "baselines", 0, request)
    if saved is None:
        started = time.perf_counter()
        raw, k, clip = item["image"], config["k"], config["clip"]
        selected = np.sort(np.random.default_rng(0).choice(
            raw.size, min(2_000_000, raw.size), replace=False,
        ))
        sample = (np.minimum(raw.ravel()[selected].astype(np.float32), clip) / clip)[:, None]
        with calculation_threads(8):
            pixel_model = KMeans(n_clusters=k, random_state=0).fit(sample)
        centers = pixel_model.cluster_centers_.ravel()
        order = np.argsort(centers)
        rank = np.empty_like(order)
        rank[order] = np.arange(k)
        pixel_labels = np.empty(raw.shape, dtype=np.uint8)
        for row in range(0, raw.shape[0], 512):
            scaled = np.minimum(raw[row:row + 512].astype(np.float32), clip) / clip
            pixel_labels[row:row + 512] = rank[np.argmin(
                np.abs(scaled[..., None] - centers[None, None, :]), axis=-1,
            )]
        means = patches[image].obsm["image_patches"].reshape(patches[image].n_obs, -1).mean(1)
        with calculation_threads(8):
            patch_labels = KMeans(n_clusters=k, random_state=0).fit_predict(means[:, None])
        thresholds = threshold_multiotsu(means, classes=k)
        otsu_labels = np.digitize(means, thresholds)
        saved = store.save(
            "intensity", "baselines", 0, request, time.perf_counter() - started,
            pixel_labels=pixel_labels, patch_labels=patch_labels,
            otsu_labels=otsu_labels, centers=np.sort(centers), thresholds=thresholds,
        )
    record_score(image, "intensity", "Pixel K-means", 0, saved["pixel_labels"], pixel=True)
    record_score(image, "intensity", "Patch-mean K-means", 0, saved["patch_labels"])
    record_score(image, "intensity", "Multi-Otsu", 0, saved["otsu_labels"])
display(pd.DataFrame(score_rows))

## Train the shared CNNs

Each image has one joint-selected CNN for each of five seeds and one
independently selected CNN for each of three seeds. The helper implements the
published convolutional architecture, augmentation and NT-Xent loss with
public PyTorch. CNN features are standardised once and shared unchanged.

In [ ]:
cnn_results = {}
for image, config in SETTINGS["images"].items():
    windows = np.asarray(patches[image].obsm["image_patches"], dtype=np.float32)
    for arm in ("joint", "matched"):
        if arm == "joint":
            selected = config["joint"]
            architecture = {name: selected["cnn_" + name]
                            for name in ("blocks", "width", "readout", "activation")}
            fixed = {name: selected["cnn_" + name] for name in (
                "latent", "projection", "temperature", "lr", "weight_decay",
                "batch", "epochs", "augment", "standardize",
            )}
        else:
            architecture = config["matched_cnn"]["architecture"]
            fixed = SETTINGS["matched_cnn_fixed"]
        for seed in SETTINGS["seeds"][arm]:
            request = {"architecture": architecture, "training": fixed}
            saved = stores[image].load(arm, "cnn", seed, request)
            if saved is None:
                started = time.perf_counter()
                cnn, features, losses, scaler, updates = fit_selected_cnn(
                    windows, architecture, fixed, seed, device=DEVICE,
                )
                saved = stores[image].save(
                    arm, "cnn", seed, request, time.perf_counter() - started,
                    metadata={"optimizer_updates": updates, "reference_labels_used": False},
                    features=features, losses=losses,
                    scaler_mean=scaler.mean_, scaler_scale=scaler.scale_,
                )
                del cnn, scaler
                release_cuda()
            cnn_results[image, arm, seed] = saved
            kmeans_settings = SETTINGS["kmeans"][arm].copy()
            if arm == "matched":
                kmeans_settings["random_state"] = seed
            with calculation_threads(2 if arm == "joint" else 4):
                labels = KMeans(n_clusters=config["k"], **kmeans_settings).fit_predict(saved["features"])
            saved["labels"] = labels
            record_score(image, arm, "CNN K-means", seed, labels)
            pd.DataFrame({"patch_id": patches[image].obs_names, "cluster": labels}).to_csv(
                RESULTS / image / arm / "cnn" / f"seed_{seed}" / "labels.csv", index=False,
            )
display(pd.DataFrame(score_rows).query("method == 'CNN K-means'"))

## Full graph, shuffled edges and no-edge control

Joint GraphMAE uses GINE, four 128-unit layers, 50% masking and the selected
graph neighbourhood. Edge attributes are standardised log-distance and
orientation. Shuffling permutes edge destinations while retaining attribute
rows; the no-edge control removes every edge. Both reuse the same CNN.

In [ ]:
graph_results = {}
for image, config in SETTINGS["images"].items():
    selected = config["joint"]
    for seed in SETTINGS["seeds"]["joint"]:
        features = cnn_results[image, "joint", seed]["features"]
        for method, control in (("graphmae", None), ("shuffled_edges", "shuffled_edges"),
                                ("no_edges", "no_edges")):
            request = {"selected": selected, "control": control,
                       "features_sha256": array_sha256(features)}
            saved = stores[image].load("joint", method, seed, request)
            if saved is None:
                started = time.perf_counter()
                seed_everything(seed, threads=2)
                data = patches[image].copy()
                del data.obsm["image_patches"]
                data.obsm["X_cnn"] = features
                edges = build_selected_graph(
                    data, selected["neighbours"], selected["continuity_thresh"],
                    control=control, seed=seed,
                )
                with observe_graphmae_training(selected["gm_updates"]) as observed:
                    model = mt.nn.encode_graphmae(
                        data, graph_key="patch", node_feature_key="X_cnn",
                        key_added="X_graphmae", hidden_dim=selected["gm_width"],
                        n_layers=selected["gm_depth"], encoder=selected["edge_use"],
                        decoder="graphmae_remask", norm="layernorm",
                        mask_ratio=selected["gm_mask"], mitigation="jk",
                        max_epochs=selected["gm_updates"], lr=selected["gm_lr"],
                        weight_decay=selected["gm_weight_decay"], recon_loss="scaled_cosine",
                        standardize=selected["gm_standardize"], random_state=seed,
                        accelerator=DEVICE, enable_progress_bar=True,
                    )
                embedding = np.asarray(data.obsm["X_graphmae"], dtype=np.float32)
                losses = np.asarray(observed["losses"])
                with calculation_threads(2):
                    labels = KMeans(n_clusters=config["k"], **SETTINGS["kmeans"]["joint"]).fit_predict(embedding)
                saved = stores[image].save(
                    "joint", method, seed, request, time.perf_counter() - started,
                    metadata={"directed_edges": edges, "reference_labels_used": False,
                              "optimizer_updates": len(observed["updates"])},
                    embedding=embedding, labels=labels, losses=losses,
                )
                del data, model
                release_cuda()
            graph_results[image, "joint", method, seed] = saved
            record_score(image, "joint", method, seed, saved["labels"])

## Native spatial comparisons

Coordinates are converted to micrometres. BANKSY and UTAG retain their native
neighbourhood transformations, followed by the specified PCA and K-means.
CellCharter uses its native mean neighbourhood aggregation and Gaussian
mixture clustering. Its published fits use CPU.

In [ ]:
from banksy.initialize_banksy import initialize_banksy
from banksy.embed_banksy import generate_banksy_matrix
from utag import utag
import cellcharter as cc
import squidpy as sq

def fit_native_comparison(image, features, method, config, seed, n_init):
    seed_everything(seed, threads=4)
    data = ad.AnnData(
        X=features.copy(),
        obs=pd.DataFrame({"sample_id": pd.Categorical([image] * len(features))},
                         index=patches[image].obs_names.copy()),
    )
    data.obsm["spatial"] = np.asarray(patches[image].obsm["spatial"]) * SETTINGS["pixel_size_um"]
    xy, k = data.obsm["spatial"], SETTINGS["images"][image]["k"]
    guard = SETTINGS["directed_edge_guard"]
    with calculation_threads(4):
        if method == "banksy":
            edges = len(features) * config["k"]
            if edges > guard:
                raise RuntimeError("Selected BANKSY setting exceeds the published edge guard.")
            data.obs["x"], data.obs["y"] = xy[:, 0], xy[:, 1]
            banksy = initialize_banksy(
                data, coord_keys=("x", "y", "spatial"),
                num_neighbours=config["k"], nbr_weight_decay="scaled_gaussian",
                max_m=config["max_m"], plt_edge_hist=False, plt_nbr_weights=False,
                plt_agf_angles=False, plt_theta=False,
            )
            banksy, _ = generate_banksy_matrix(
                data, banksy, lambda_list=[config["lambda"]],
                max_m=config["max_m"], verbose=False,
            )
            transformed = banksy["scaled_gaussian"][config["lambda"]]["adata"].X
            transformed = transformed.toarray() if sparse.issparse(transformed) else np.asarray(transformed)
            embedding = PCA(n_components=config["pca"], random_state=seed).fit_transform(
                transformed.astype(np.float32),
            )
            labels = KMeans(n_clusters=k, n_init=n_init, random_state=seed).fit_predict(embedding)
        elif method == "utag":
            edges = int(cKDTree(xy).query_ball_point(
                xy, config["radius_um"], return_length=True,
            ).sum())
            if edges > guard:
                raise RuntimeError("Selected UTAG setting exceeds the published edge guard.")
            transformed = utag(
                data, slide_key=None, max_dist=config["radius_um"],
                normalization_mode=config["normalization"],
                apply_clustering=False, parallel=False,
            ).X
            transformed = transformed.toarray() if sparse.issparse(transformed) else np.asarray(transformed)
            embedding = PCA(n_components=config["pca"], random_state=seed).fit_transform(
                transformed.astype(np.float32),
            )
            labels = KMeans(n_clusters=k, n_init=n_init, random_state=seed).fit_predict(embedding)
        else:
            if config["graph"] == "native_knn6":
                sq.gr.spatial_neighbors(
                    data, coord_type="generic", n_neighs=6, library_key="sample_id",
                )
                connectivity = "spatial_connectivities"
            else:
                neighbours = int(config["graph"].split("_k")[1])
                geometry = patches[image].copy()
                del geometry.obsm["image_patches"]
                mt.gr.build_patch_graph(
                    geometry, k=neighbours, continuity_thresh=.25,
                    continuity_samples=6, key_added="patch",
                )
                edges = np.asarray(geometry.uns["patch"]["edge_index"])
                connectivity = "mantpy_connectivities"
                data.obsp[connectivity] = sparse.csr_matrix(
                    (np.ones(edges.shape[1]), (edges[0], edges[1])),
                    shape=(len(features), len(features)),
                )
                del geometry
            edges = int(data.obsp[connectivity].nnz)
            if edges > guard:
                raise RuntimeError("Selected CellCharter setting exceeds the published edge guard.")
            cc.gr.aggregate_neighbors(
                data, n_layers=config["hops"], aggregations="mean",
                connectivity_key=connectivity, sample_key="sample_id",
            )
            estimator = cc.tl.Cluster(
                n_clusters=k, covariance_type="full", random_state=seed,
                trainer_params={"accelerator": "cpu"},
            )
            estimator.fit(data)
            labels = np.asarray(estimator.predict(data), dtype=np.int64)
            embedding = np.asarray(data.obsm["X_cellcharter"])
    assert labels.shape == (len(features),) and np.isfinite(embedding).all()
    return labels, np.asarray(embedding, dtype=np.float32), edges

### Published default comparisons

Seed-0 joint CNN: BANKSY uses 15 neighbours, λ=0.2 and PCA-20; UTAG uses a
20 µm radius and PCA-10; CellCharter uses six neighbours and three hops.
These defaults use K-means `n_init="auto"`. CNN Ward is also included.

In [ ]:
native_results = {}
for image, config in SETTINGS["images"].items():
    features = cnn_results[image, "joint", 0]["features"]
    default_configurations = {
        "banksy": {"k": 15, "lambda": .2, "max_m": 1, "pca": 20},
        "utag": {"radius_um": 20., "pca": 10, "normalization": "l1_norm"},
        "cellcharter": {"graph": "native_knn6", "hops": 3},
    }
    for method, selected in default_configurations.items():
        request = {"config": selected, "n_init": "auto",
                   "features_sha256": array_sha256(features)}
        saved = stores[image].load("defaults", method, 0, request)
        if saved is None:
            started = time.perf_counter()
            labels, embedding, edges = fit_native_comparison(
                image, features, method, selected, 0, n_init="auto",
            )
            saved = stores[image].save(
                "defaults", method, 0, request, time.perf_counter() - started,
                metadata={"directed_edges": edges, "reference_labels_used": False},
                labels=labels, embedding=embedding,
            )
        native_results[image, "defaults", method, 0] = saved
        record_score(image, "defaults", method, 0, saved["labels"])
    request = {"linkage": "ward", "standardize": True,
               "features_sha256": array_sha256(features)}
    saved = stores[image].load("defaults", "cnn_ward", 0, request)
    if saved is None:
        started = time.perf_counter()
        labels = AgglomerativeClustering(n_clusters=config["k"], linkage="ward").fit_predict(
            StandardScaler().fit_transform(features).astype(np.float32),
        )
        saved = stores[image].save(
            "defaults", "cnn_ward", 0, request, time.perf_counter() - started, labels=labels,
        )
    record_score(image, "defaults", "CNN Ward", 0, saved["labels"])
display(pd.DataFrame(score_rows).query("arm == 'defaults'"))

### Independently selected matched comparisons

These are the deposited per-method selections, made without reference
annotations. The twelve candidate settings were searched in the paper;
this notebook refits only the selected setting for each method.

In [ ]:
selected_rows = []
for image, config in SETTINGS["images"].items():
    for method, record in config["matched_methods"].items():
        selected_rows.append({
            "image": image, "method": method,
            "selection_index": record["index"], "config": record["config"],
        })
display(pd.DataFrame(selected_rows))

for image, config in SETTINGS["images"].items():
    for seed in SETTINGS["seeds"]["matched"]:
        features = cnn_results[image, "matched", seed]["features"]
        for method in ("graphmae", "banksy", "utag", "cellcharter"):
            selected = config["matched_methods"][method]["config"]
            request = {"selected": selected, "features_sha256": array_sha256(features),
                       "n_init": 10, "training": SETTINGS["matched_graphmae_fixed"] if method == "graphmae" else None}
            saved = stores[image].load("matched", method, seed, request)
            if saved is None:
                started = time.perf_counter()
                if method == "graphmae":
                    seed_everything(seed, threads=4)
                    data = patches[image].copy()
                    del data.obsm["image_patches"]
                    data.obsm["X_cnn"] = features
                    edges = build_selected_graph(data, selected["k"])
                    if edges > SETTINGS["directed_edge_guard"]:
                        raise RuntimeError("Selected GraphMAE setting exceeds the published edge guard.")
                    with observe_graphmae_training(200) as observed:
                        model = mt.nn.encode_graphmae(
                            data, graph_key="patch", node_feature_key="X_cnn",
                            key_added="X_graphmae", n_layers=selected["depth"],
                            mask_ratio=selected["mask"], random_state=seed,
                            accelerator=DEVICE, enable_progress_bar=True,
                            **SETTINGS["matched_graphmae_fixed"],
                        )
                    embedding = np.asarray(data.obsm["X_graphmae"], dtype=np.float32)
                    losses = np.asarray(observed["losses"])
                    with calculation_threads(4):
                        labels = KMeans(n_clusters=config["k"], n_init=10,
                                        random_state=seed).fit_predict(embedding)
                    del data, model
                    release_cuda()
                else:
                    labels, embedding, edges = fit_native_comparison(
                        image, features, method, selected, seed, n_init=10,
                    )
                    losses = np.empty(0)
                saved = stores[image].save(
                    "matched", method, seed, request, time.perf_counter() - started,
                    metadata={"directed_edges": edges, "reference_labels_used": False,
                              "optimizer_updates": len(observed["updates"]) if method == "graphmae" else None},
                    labels=labels, embedding=embedding, losses=losses,
                )
            if method == "graphmae":
                graph_results[image, "matched", method, seed] = saved
            else:
                native_results[image, "matched", method, seed] = saved
            record_score(image, "matched", method, seed, saved["labels"])

## Compare agreement and inspect representative maps

ARI is invariant to cluster numbering. Macro-F1 uses one-to-one cluster/class
matching. Full-area scores include one unassigned label outside the same
foreground support for every method. Means and sample SDs summarise seeds;
single-fit baselines have no seed SD.

In [ ]:
scores, summary = score_summary(score_rows)
display(summary.round(3))
for image in SETTINGS["images"]:
    comparison = summary.query("image == @image and arm in ['joint', 'matched']")
    figure, axis = plt.subplots(figsize=(8, 5))
    positions = np.arange(len(comparison))
    axis.errorbar(comparison["ARI_mean"], positions,
                  xerr=comparison["ARI_std"].fillna(0), fmt="o", capsize=3)
    axis.set_yticks(positions, comparison["arm"] + ": " + comparison["method"])
    axis.set(xlabel="ARI on covered annotated pixels", title=image, xlim=(-.05, 1))
    axis.invert_yaxis()
    figure.tight_layout()
    figure.savefig(RESULTS / image / "agreement.png", dpi=160)
    plt.show()
    plot_patch_labels(
        patches[image], cnn_results[image, "joint", 0]["labels"],
        f"{image}: joint CNN K-means, seed 0", SETTINGS["images"][image]["k"],
        save=RESULTS / image / "cnn_seed0.png",
    )
    plot_patch_labels(
        patches[image], graph_results[image, "joint", "graphmae", 0]["labels"],
        f"{image}: joint GraphMAE, seed 0", SETTINGS["images"][image]["k"],
        save=RESULTS / image / "graphmae_seed0.png",
    )

## Supplementary: GIN versus GINE at the joint-selected setting

Refit GIN with the same CNN features, topology and training settings for all
five seeds. GINE is already fitted above. GIN ignores the edge attributes.

In [ ]:
for image, config in SETTINGS["images"].items():
    selected = config["joint"]
    for seed in SETTINGS["seeds"]["joint"]:
        features = cnn_results[image, "joint", seed]["features"]
        request = {"selected": selected, "encoder": "gin",
                   "features_sha256": array_sha256(features)}
        saved = stores[image].load("supplementary", "gin", seed, request)
        if saved is None:
            started = time.perf_counter()
            seed_everything(seed, threads=2)
            data = patches[image].copy()
            del data.obsm["image_patches"]
            data.obsm["X_cnn"] = features
            edges = build_selected_graph(data, selected["neighbours"])
            with observe_graphmae_training(200) as observed:
                model = mt.nn.encode_graphmae(
                    data, graph_key="patch", node_feature_key="X_cnn", key_added="X_graphmae",
                    hidden_dim=128, n_layers=4, encoder="gin", decoder="graphmae_remask",
                    norm="layernorm", mask_ratio=.5, mitigation="jk", max_epochs=200,
                    lr=.001, weight_decay=.0001, recon_loss="scaled_cosine",
                    standardize=False, random_state=seed, accelerator=DEVICE,
                    enable_progress_bar=True,
                )
            embedding = np.asarray(data.obsm["X_graphmae"], dtype=np.float32)
            with calculation_threads(2):
                labels = KMeans(n_clusters=config["k"], **SETTINGS["kmeans"]["joint"]).fit_predict(embedding)
            saved = stores[image].save(
                "supplementary", "gin", seed, request, time.perf_counter() - started,
                metadata={"directed_edges": edges, "optimizer_updates": len(observed["updates"])},
                embedding=embedding, labels=labels, losses=np.asarray(observed["losses"]),
            )
            del data, model
            release_cuda()
        graph_results[image, "supplementary", "gin", seed] = saved
        record_score(image, "supplementary", "GIN", seed, saved["labels"])
display(score_summary(score_rows)[1].query("method in ['GIN', 'graphmae']"))

## Supplementary: vary K on the fixed embeddings

Recluster K=2–8 using the already fitted joint CNN and full-graph embeddings.
Report pairwise seed agreement and minimum cluster fraction. This changes
only clustering and does not retrain encoders or select a new paper setting.

In [ ]:
from itertools import combinations
from sklearn.metrics import adjusted_rand_score

k_rows = []
for image in SETTINGS["images"]:
    for method in ("CNN", "GraphMAE"):
        for k in range(2, 9):
            labels_by_seed, fractions = [], []
            for seed in SETTINGS["seeds"]["joint"]:
                embedding = (cnn_results[image, "joint", seed]["features"] if method == "CNN"
                             else graph_results[image, "joint", "graphmae", seed]["embedding"])
                with calculation_threads(2):
                    labels = KMeans(n_clusters=k, **SETTINGS["kmeans"]["joint"]).fit_predict(embedding)
                labels_by_seed.append(labels)
                fractions.append(float(np.bincount(labels, minlength=k).min() / len(labels)))
            agreement = [adjusted_rand_score(a, b) for a, b in combinations(labels_by_seed, 2)]
            k_rows.append({"image": image, "method": method, "K": k,
                           "seed_ARI": float(np.mean(agreement)),
                           "minimum_cluster_fraction": min(fractions)})
k_sweep = pd.DataFrame(k_rows)
display(k_sweep.round(3))
for image in SETTINGS["images"]:
    figure, axis = plt.subplots(figsize=(6, 4))
    for method, group in k_sweep.query("image == @image").groupby("method"):
        axis.plot(group["K"], group["seed_ARI"], "o-", label=method)
    axis.axvline(SETTINGS["images"][image]["k"], color="gray", linestyle=":")
    axis.set(xlabel="K", ylabel="Mean pairwise seed ARI", title=image)
    axis.legend()
    figure.tight_layout()
    plt.show()

## Archived selection records

These tables describe the paper's searches. They are references, separate
from the fresh fits above, and are never used to choose a setting in this run.
Exact published figure source data and plotting code remain in the repository.

In [ ]:
for image, config in SETTINGS["images"].items():
    print(image, "— archived independent CNN search")
    display(pd.DataFrame(config["matched_cnn_search_table"])[
        ["architecture_index", "architecture", "eligible", "stability"]
    ])
    for method, records in config["matched_method_search_tables"].items():
        print(method)
        display(pd.DataFrame(records))

## Export results and measured runtime

Exports contain fresh seed-level scores, summary tables, K-sweep results,
label arrays, feature arrays, training losses, settings and stage times.
Re-running this notebook verifies checkpoint identities before reusing fits.
Differences across hardware/software can affect nearest-neighbour ties and
stochastic training; exact deposited paper scores remain available as source data.

In [ ]:
scores, summary = score_summary(score_rows)
scores.to_csv(RESULTS / "scores.csv", index=False)
summary.to_csv(RESULTS / "score_summary.csv", index=False)
k_sweep.to_csv(RESULTS / "k_sweep.csv", index=False)
write_json(RESULTS / "settings.json", SETTINGS)
timings = pd.concat([
    store.save_timing().assign(image=image) for image, store in stores.items()
], ignore_index=True)
timings.to_csv(RESULTS / "timings.csv", index=False)
measured_seconds = time.perf_counter() - NOTEBOOK_STARTED
write_json(RESULTS / "run_summary.json", {
    "notebook_seconds": measured_seconds,
    "completed_fit_seconds": float(timings["seconds"].sum()),
    "reused_stages": int(timings["reused"].sum()),
    "completed_stages": len(timings),
    "environment": environment(),
    "configuration_searches": 0,
})
display(timings.groupby(["image", "arm"])["seconds"].sum().to_frame())
print(f"Notebook elapsed: {measured_seconds / 60:.1f} min")
print("Outputs:", RESULTS.resolve())